# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 15

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В датафрейме столбец `registered_at` хранит даты регистрации клиентов (например, `'2025-05-12'`), но имеет тип данных `object` (текстовая строка).
Сможет ли аналитик напрямую вычислить разницу в днях между датами (например, `df['registered_at'] - datetime.now()`)? Какая функция Pandas преобразует столбец в специальный тип даты/времени?

**Ответ:**
Нет, со строками типа `object` математические операции дат вызовут ошибку `TypeError`. Для перевода строк в дату/время используется функция `pd.to_datetime(df['registered_at'])`.

---

#### 1.2. Аналитик применил метод `df['balance'].describe()` к столбцу баланса клиентов и получил:
`mean = 1450.0`, `50% (медиана) = 500.0`, `max = 7500.0`.
О чем говорит такой сильный разрыв между средним арифметическим (1450) и медианой (500)? Почему медиана лучше отражает типичный баланс клиента клуба?

**Ответ:**
Разрыв говорит о наличии небольшого числа клиентов с очень крупным балансом (выбросы/VIP-клиенты), которые сильно завышают среднее арифметическое. Медиана (500 руб.) показывает, что ровно у половины клиентов баланс не превышает 500 руб., поэтому она лучше характеризует типичного посетителя.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["is_marathon"] = df["duration_hours"] >= 4.0
```

**Ответ:**
Формирует булев флаг длинной («марафонской») игровой сессии продолжительностью от 4 часов и более.

---

#### 1.4. Почему следующий код фильтрации в Pandas упадет с ошибкой или сработает неверно?
```python
# Ошибочный код:
df[df['balance'] > 1000 & df['city'] == 'Москва']
```
Как правильно записать это условие со скобками и почему круглые скобки обязательны?

**Ответ:**
В Python побитовый оператор `&` имеет более высокий приоритет, чем операции сравнения `>` и `==`. Без скобок интерпретатор пытается вычислить `1000 & df['city']`. Правильная запись: `df[(df['balance'] > 1000) & (df['city'] == 'Москва')]`.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `tournaments.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("tournaments.csv").parent if os.path.exists("tournaments.csv") else pathlib.Path("../variant_15")
df_audit = pd.read_csv(data_dir / "tournaments.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Минимальный призовой фонд: 52500.00")

Количество строк в таблице: 8
Столбцы датафрейма: ['tournament_id', 'tournament_name', 'game_title', 'prize_fund', 'entry_fee', 'start_date', 'min_rank']
Минимальный призовой фонд: 52500.00

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_15")
df_data = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

summary = df_data.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

summary["quantity"] = summary["quantity"].round(2)
summary = summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(summary)

,category,total_amount,quantity
0,Мерч,110700,1.40
1,Горячее,36820,1.67
2,Напитки,28540,1.50
3,Снеки,14120,1.40


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `hardware_pcs` столбец `pc_number` (например, 'PC-01') имеет ограничение `UNIQUE`.
От какой проблемы учета это защищает клуб, и может ли столбец с ограничением `UNIQUE` содержать значение `NULL`?

**Ответ:**
Ограничение `UNIQUE` гарантирует, что в клубе не появятся два компьютера с одинаковым инвентарным номером 'PC-01'. В отличие от `PRIMARY KEY`, столбец с ограничением `UNIQUE` в SQL может содержать значение `NULL` (отсутствие номера).

---

#### 3.2. Разработчик через скрипт на Python выполнил добавление новых компьютеров командой `cursor.execute("INSERT INTO hardware_pcs ...")`, но после закрытия программы обнаружил, что компьютеры не сохранились в файле базы данных.
Какую обязательную команду для объекта соединения (`conn`) он забыл вызвать и почему?

**Ответ:**
Он забыл вызвать команду фиксации транзакции `conn.commit()`. В библиотеке `sqlite3` изменения, внесенные через курсор, по умолчанию находятся во временной транзакции и не сохраняются на диске без явного `commit()`.

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT min_rank, COUNT(*) AS tourn_cnt, SUM(prize_fund) AS total_prizes FROM tournaments GROUP BY min_rank ORDER BY total_prizes DESC;
```

**Ответ:**
Запрос группирует турниры по минимальному рангу участников (`min_rank`) и рассчитывает количество турниров и суммарный призовой фонд, упорядочивая по убыванию призовых.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_15")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT order_id, member_id, item_name, category, item_price, total_amount
FROM bar_orders
WHERE category IN ('Напитки', 'Снеки') AND item_price <= 150
ORDER BY total_amount DESC, order_id ASC
LIMIT 8;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,order_id,member_id,item_name,category,item_price,total_amount
0,6131,53,Сухарики чесночные,Снеки,110.0,440.0
1,6091,41,Минеральная вода Borjomi,Напитки,140.0,420.0
2,6263,139,Минеральная вода Borjomi,Напитки,140.0,420.0
3,6267,106,Минеральная вода Borjomi,Напитки,140.0,420.0
4,6073,117,Орешки арахис соленый,Снеки,130.0,390.0
5,6186,96,Орешки арахис соленый,Снеки,130.0,390.0
6,6092,77,Чай зеленый с мятой,Напитки,120.0,360.0
7,6096,115,Чай зеленый с мятой,Напитки,120.0,360.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT item_name, COUNT(*) AS orders_count, ROUND(SUM(total_amount), 2) AS revenue
FROM bar_orders
GROUP BY item_name
HAVING SUM(total_amount) > 5000
ORDER BY revenue DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,item_name,orders_count,revenue
0,Мерч-футболка CyberClub,25,55500.0
1,Коврик для мыши CyberClub Speed,23,28800.0
2,Кепка CyberClub Team,25,26400.0
3,Бургер Классический,27,18020.0
4,Наггетсы куриные (8 шт),26,11000.0
5,Энергетик Adrenaline Rush,30,9880.0
6,Сэндвич с ветчиной и сыром,23,7800.0
7,Минеральная вода Borjomi,30,5740.0
8,Кофе Латте,18,5220.0


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, COUNT(b.order_id) AS snacks_count, ROUND(SUM(b.total_amount), 2) AS snacks_spent
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
WHERE b.category = 'Снеки'
GROUP BY m.member_id, m.nickname
ORDER BY snacks_spent DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,snacks_count,snacks_spent
0,49,Paladin_874,3,810.0
1,117,Shadow_335,3,710.0
2,96,Sentinel_368,2,485.0
3,119,Frost_103,3,485.0
4,88,Vortex_222,2,480.0
5,53,Vortex_589,1,440.0
6,55,Sentinel_787,2,430.0
7,130,Matrix_430,2,415.0
8,54,Turbo_690,3,380.0
9,124,Phantom_417,2,355.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT order_id, member_id, item_name, category, total_amount
FROM bar_orders
WHERE total_amount > (SELECT AVG(total_amount) FROM bar_orders WHERE category = 'Напитки')
ORDER BY total_amount DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,order_id,member_id,item_name,category,total_amount
0,6295,140,Мерч-футболка CyberClub,Мерч,6000.0
1,6243,120,Мерч-футболка CyberClub,Мерч,4500.0
2,6129,65,Коврик для мыши CyberClub Speed,Мерч,3600.0
3,6197,50,Коврик для мыши CyberClub Speed,Мерч,3600.0
4,6011,98,Мерч-футболка CyberClub,Мерч,3000.0
5,6026,112,Мерч-футболка CyberClub,Мерч,3000.0
6,6070,97,Мерч-футболка CyberClub,Мерч,3000.0
7,6154,42,Мерч-футболка CyberClub,Мерч,3000.0
8,6155,58,Мерч-футболка CyberClub,Мерч,3000.0
9,6232,108,Мерч-футболка CyberClub,Мерч,3000.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Длительные игровые сессии на компьютерах

**Найденные ошибки:**
1. Ошибка ключа соединения: условие `ON p.pc_id = s.member_id` связывает идентификатор компьютера с ID клиента вместо `s.pc_id`.

In [9]:
query_51_fixed = '''
SELECT 
    p.pc_number,
    p.zone_name,
    s.game_title,
    s.duration_hours
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE s.duration_hours >= 3
ORDER BY s.duration_hours DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,pc_number,zone_name,game_title,duration_hours
0,PC-17,Bootcamp,Dota 2,6.0
1,PC-05,Standard,Cyberpunk 2077,6.0
2,PC-20,Bootcamp,Warzone,6.0
3,PC-15,Bootcamp,Atomic Heart,6.0
4,PC-15,Bootcamp,Counter-Strike 2,6.0
5,PC-16,Bootcamp,Atomic Heart,6.0
6,PC-14,Standard,GTA V,6.0
7,PC-09,Standard,Dota 2,6.0
8,PC-16,Bootcamp,PUBG,5.0
9,PC-06,Standard,Atomic Heart,5.0


#### 5.2. Средний чек по городам клиентов

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает клиента с компьютером вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: фильтрация по `AVG(s.total_paid) > 300` должна выполняться в секции `HAVING`.

In [10]:
query_52_fixed = '''
SELECT 
    m.city,
    COUNT(s.session_id) AS sessions_cnt,
    ROUND(AVG(s.total_paid), 2) AS avg_paid
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.city
HAVING AVG(s.total_paid) > 300
ORDER BY avg_paid DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,city,sessions_cnt,avg_paid
0,Москва,202,494.60
1,Казань,56,486.16
2,Санкт-Петербург,135,469.49
3,Новосибирск,66,438.61
4,Екатеринбург,41,372.76


#### 5.3. Сравнение призовых фондов турниров со средним фондом

**Найденные ошибки:**
1. Неполный `GROUP BY`: поле `min_rank` указано в `SELECT`, но не включено в список группировки.
2. Порядок предложений и alias: секция `WHERE` написана после `GROUP BY` и использует псевдоним `avg_prize` вместо `HAVING AVG(prize_fund)`.
3. Многострочный подзапрос: наличие `GROUP BY game_title` в подзапросе возвращает несколько строк вместо одного скалярного значения.

In [11]:
query_53_fixed = '''
SELECT 
    game_title,
    min_rank,
    ROUND(AVG(prize_fund), 2) AS avg_prize
FROM tournaments
GROUP BY game_title, min_rank
HAVING AVG(prize_fund) > (
    SELECT AVG(prize_fund) 
    FROM tournaments
)
ORDER BY avg_prize DESC;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,game_title,min_rank,avg_prize
0,Counter-Strike 2,Gold,120000.0
1,Dota 2,Silver,75000.0
